In [ ]:
# The statistics behind the figures, as tables. Every test is a Holm-corrected
# paired Wilcoxon on one value per allele or per withheld molecule.
import figio as fx
import pandas as pd

def report(title, df, unit, value):
    res = fx.paired_test(df, unit=unit, value=value,
                         models=df['full_name'].drop_duplicates().tolist())
    print(f'===== {title} =====')
    print(res[['a', 'b', 'n', 'median_delta', 'p', 'p_adj', 'sig']]
          .to_string(index=False, float_format=lambda v: f'{v:.4g}'))
    print()
    return res

whole = fx.by_beta(fx.alleles('1_whole'))
ms = fx.by_beta(fx.alleles('2_ms'))
lomo_beta = fx.by_beta(fx.lomo())

r1 = report('Qualitative, allele-wise (beta-collapsed)', whole, 'beta', 'roc_auc')
r2 = report('MS, allele-wise (beta-collapsed)', ms, 'beta', 'roc_auc')
r3 = report('LOMO, per withheld molecule (beta-collapsed)', lomo_beta, 'beta', 'roc_auc')

In [ ]:
# Sensitivity: does the beta collapse drive anything? Two alternatives.
#   B  the full 47 alpha/beta pairs, DeepNeo necessarily excluded
#   C  only the beta chains carrying exactly one alpha pairing, so nothing is averaged
lomo_raw = fx.lomo()
pairs = lomo_raw[lomo_raw['model'] != 'deepneo']
rB = report('LOMO, all 47 pairs, pair-split models only', pairs, 'molecule', 'roc_auc')

one2one = lomo_beta.loc[lomo_beta['n_pairs'] == 1, 'beta'].value_counts()
keep = [b for b, c in one2one.items() if c == lomo_beta['model'].nunique()]
rC = report(f'LOMO, one-to-one betas only (n={len(keep)})',
            lomo_beta[lomo_beta['beta'].isin(keep)], 'beta', 'roc_auc')

# The three should agree on every conclusion; if they do, the collapse is not
# doing the work and a reviewer asking about it has an answer in one table.
merged = (r3[['a', 'b', 'sig']].rename(columns={'sig': 'A_beta_n38'})
          .merge(rB[['a', 'b', 'sig']].rename(columns={'sig': 'B_pairs_n47'}), on=['a', 'b'], how='left')
          .merge(rC[['a', 'b', 'sig']].rename(columns={'sig': 'C_one2one'}), on=['a', 'b'], how='left'))
merged

In [ ]:
# Median and IQR per representation, which is what the allele-wise and LOMO
# paragraphs quote alongside the tests.
for title, df, value in [('Qualitative alleles', whole, 'roc_auc'),
                         ('MS alleles', ms, 'roc_auc'),
                         ('LOMO molecules', lomo_beta, 'roc_auc')]:
    g = df.groupby('full_name')[value]
    q = g.quantile([.25, .75]).unstack()
    out = pd.DataFrame({'median': g.median(), 'IQR': q[0.75] - q[0.25], 'n': g.size()})
    print(f'--- {title} ---')
    print(out.sort_values('median', ascending=False).round(4).to_string())
    print()